# 1. Setup môi trường
Cài thư viện còn thiếu, tải tài nguyên NLTK và đưa dataset
[Fake and Real News (Kaggle)](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) vào `data/raw/`.
Chỉ cần chạy **một lần**; các notebook 2 → 5 dùng lại kết quả.

In [1]:
import sys, subprocess, importlib.util

PACKAGES = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib",
            "nltk": "nltk", "sklearn": "scikit-learn", "kagglehub": "kagglehub"}
missing = [pkg for mod, pkg in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Cài thêm:", missing or "không cần")

Cài thêm: không cần


In [2]:
import nltk

# stop-word list, từ điển WordNet (lemmatization) và bộ gắn nhãn từ loại (POS) cho lemmatization chính xác hơn
for resource in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)
print("NLTK sẵn sàng")

NLTK sẵn sàng


### Tải dataset
Nếu không tải được qua KaggleHub (lỗi mạng), tải file zip bằng tay rồi đặt vào `data/upload/`.

In [3]:
import shutil, zipfile
from pathlib import Path
import pandas as pd

RAW_DIR, UPLOAD_DIR = Path("data/raw"), Path("data/upload")
FILES = ["Fake.csv", "True.csv"]
RAW_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

def copy_csvs(folder):
    for f in FILES:
        shutil.copy2(next(Path(folder).rglob(f)), RAW_DIR / f)

if not all((RAW_DIR / f).exists() for f in FILES):
    for z in UPLOAD_DIR.glob("*.zip"):
        zipfile.ZipFile(z).extractall(UPLOAD_DIR)
    if all(any(UPLOAD_DIR.rglob(f)) for f in FILES):
        copy_csvs(UPLOAD_DIR)
    else:
        import kagglehub
        copy_csvs(kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset"))

for f in FILES:
    d = pd.read_csv(RAW_DIR / f)
    print(f"{f}: {len(d):,} bài | cột: {list(d.columns)}")

Fake.csv: 23,481 bài | cột: ['title', 'text', 'subject', 'date']


True.csv: 21,417 bài | cột: ['title', 'text', 'subject', 'date']
